# Task
Generate a Python script for Google Colab that implements and trains a MambaMixture-of-Experts (MoE) model for sentiment classification on the SST-2 dataset.

The script should include:
1.  **Environment Setup**: Install `transformers`, `datasets`, and a Mamba-SSM implementation.
2.  **Data Loading and Preprocessing**: Load the SST-2 dataset and `bert-base-uncased` tokenizer. Preprocess the data with `max_length=128`.
3.  **DataLoaders**: Create PyTorch DataLoaders with `batch_size=32` for training and evaluation.
4.  **Pseudo-labeling for CoT**: Derive a pseudo-label based on input sequence length (e.g., short/long based on `max_length`) for the intermediate "thought" head.
5.  **Mamba & MoE Architecture**: Implement Mamba blocks (`d_model=128`, `num_layers=4`, `d_state=32`) and an MoE layer with a gating network, `TopKRouter` (selecting `k=2` experts), and `N=4` Expert FFNs.
6.  **`MambaMoEClassifier` Model**: Define a custom PyTorch model combining Mamba blocks, the MoE layer (after the second Mamba block), an intermediate "thought" classification head (for pseudo-labels), and a final "answer" classification head (for sentiment).
7.  **Training Utilities**: Set up `AdamW` optimizer (`learning_rate=5e-5`), loss functions (main, CoT, and MoE load-balancing with `α=0.2`, `β=0.01`).
8.  **Low-Precision Training**: Implement Automatic Mixed Precision (AMP/FP16) using PyTorch's `autocast` and `GradScaler`.
9.  **Training and Evaluation Loop**: Train the model for `num_epochs=3`, including forward/backward passes, loss calculation, and parameter updates with AMP.
10. **Metrics**: Evaluate the model on the test set, reporting Final Classification Accuracy, F1-Score (Macro), and an MoE Load Balancing Metric (e.g., Expert Utilization Rate).
11. **Instructions**: Provide detailed instructions for running the script in a Google Colab T4 GPU environment, including how to enable GPU acceleration and verify AMP usage.

## Setup Environment and Data Loading

### Subtask:
Install necessary libraries, load and preprocess the SST-2 dataset, create PyTorch DataLoaders, and generate pseudo-labels for the Chain-of-Thought (CoT) task.


**Reasoning**:
The first step of the instructions is to install the required Python packages. I will use `pip install` within a `code_block` to install `transformers`, `datasets`, and `mamba-ssm`.



In [ ]:
pip install transformers datasets /content/mamba_ssm-2.2.6.post3-cp312-cp312-linux_x86_64.whl

Processing ./mamba_ssm-2.2.6.post3-cp312-cp312-linux_x86_64.whl
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)


## BANGER




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba

# 1. Define Mamba Block
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv=4):
        super().__init__()
        # Explicitly setting d_conv=4 as per the request
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv)

    def forward(self, x):
        return self.mamba(x)

# 2. Define Expert FFN
class ExpertFFN(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        d_ff = d_model * expansion_factor # Expansion factor of 4 as requested
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

# 3. Define TopKRouter for MoE
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, k):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.softmax = nn.Softmax(dim=-1)
        self.k = k

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        gate_logits = self.gate(x)  # (batch_size, seq_len, num_experts)
        gate_weights = self.softmax(gate_logits)

        # Select top-k experts
        top_k_weights, top_k_indices = torch.topk(gate_weights, self.k, dim=-1)
        # top_k_weights shape: (batch_size, seq_len, k)
        # top_k_indices shape: (batch_size, seq_len, k)

        # Normalize top-k weights to sum to 1
        top_k_weights_norm = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        return top_k_weights_norm, top_k_indices, gate_logits

# 4. Define MoE Layer
class MoELayer(nn.Module):
    def __init__(self, d_model, num_experts, k_experts):
        super().__init__()
        self.router = TopKRouter(d_model, num_experts, k_experts)
        # Experts with expansion factor of 4
        self.experts = nn.ModuleList([ExpertFFN(d_model, expansion_factor=4) for _ in range(num_experts)])
        self.d_model = d_model
        self.num_experts = num_experts
        self.k_experts = k_experts

    def forward(self, x, attention_mask=None):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # Get routing weights and indices
        top_k_weights_norm, top_k_indices, gate_logits = self.router(x)

        # Initialize output tensor
        output = torch.zeros_like(x, device=x.device)

        # Flatten x, top_k_weights_norm, top_k_indices for easier expert dispatch
        x_flat = x.view(-1, self.d_model)
        top_k_weights_norm_flat = top_k_weights_norm.view(-1, self.k_experts)
        top_k_indices_flat = top_k_indices.view(-1, self.k_experts)

        # Create a buffer for expert outputs
        expert_outputs_buffer = torch.zeros_like(x_flat)

        for i, expert_ffn in enumerate(self.experts):
            # Create a mask for tokens routed to expert i
            # A token is routed to expert i if i is in its top_k_indices
            expert_mask_for_current_expert = (top_k_indices_flat == i).any(dim=-1)

            if expert_mask_for_current_expert.any():
                tokens_for_expert = x_flat[expert_mask_for_current_expert]
                expert_output = expert_ffn(tokens_for_expert)

                # Get the weights corresponding to expert i for the tokens routed to it
                # This requires finding where 'i' appears in top_k_indices_flat and getting the corresponding weight
                indices_of_expert_i = (top_k_indices_flat[expert_mask_for_current_expert] == i).nonzero(as_tuple=True)
                assigned_weights = top_k_weights_norm_flat[expert_mask_for_current_expert][indices_of_expert_i[0], indices_of_expert_i[1]]

                # Accumulate weighted expert output
                expert_outputs_buffer[expert_mask_for_current_expert] += expert_output * assigned_weights.unsqueeze(-1)

        output = expert_outputs_buffer.view(batch_size, seq_len, self.d_model)

        return output, gate_logits, top_k_indices


# 5. Define MambaMoEClassifier Model
class MambaMoEClassifier(nn.Module):
    def __init__(self, d_model, num_layers, d_state, d_conv, num_experts, k_experts, num_labels_sentiment, num_labels_cot, vocab_size, max_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        # Pass d_conv to MambaBlock
        self.mamba_blocks = nn.ModuleList([MambaBlock(d_model, d_state, d_conv=d_conv) for _ in range(num_layers)])

        # Insert MoE layer after the second Mamba block (index 1 for 0-indexed list)
        self.moe_layer_index = 1
        self.moe_layer = MoELayer(d_model, num_experts, k_experts)

        self.sentiment_head = nn.Linear(d_model, num_labels_sentiment)
        self.cot_head = nn.Linear(d_model, num_labels_cot)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape

        # Embeddings
        x = self.embedding(input_ids)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = x + self.position_embedding(positions)

        # Mamba Blocks and MoE
        raw_gate_logits = None
        top_k_indices = None
        for i, block in enumerate(self.mamba_blocks):
            x = block(x) # x shape: (batch_size, seq_len, d_model)
            if i == self.moe_layer_index:
                x, raw_gate_logits, top_k_indices = self.moe_layer(x, attention_mask)

        # Global Average Pooling for classification heads
        # Only average over unmasked tokens
        if attention_mask is not None:
            # Expand attention mask to match d_model
            expanded_attention_mask = attention_mask.unsqueeze(-1).expand_as(x).float()
            masked_x = x * expanded_attention_mask
            # Sum along sequence dimension and divide by count of unmasked tokens
            pooled_output = masked_x.sum(dim=1) / expanded_attention_mask.sum(dim=1).clamp(min=1e-9) # Add clamp to avoid division by zero
        else:
            # If no attention mask, just average over sequence length
            pooled_output = x.mean(dim=1)

        # Classification heads
        sentiment_logits = self.sentiment_head(pooled_output)
        cot_logits = self.cot_head(pooled_output)

        return cot_logits, sentiment_logits, raw_gate_logits, top_k_indices

print("MambaBlock, ExpertFFN, TopKRouter, MoELayer, and MambaMoEClassifier classes defined with updated parameters.")

MambaBlock, ExpertFFN, TopKRouter, MoELayer, and MambaMoEClassifier classes defined with updated parameters.


### Define Model Parameters, Load Data and Tokenizer

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba

# 1. Define Mamba Block
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv=4):
        super().__init__()
        # Explicitly setting d_conv=4 as per the request
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv)

    def forward(self, x):
        return self.mamba(x)

# 2. Define Expert FFN
class ExpertFFN(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        d_ff = d_model * expansion_factor # Expansion factor of 4 as requested
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

# 3. Define TopKRouter for MoE
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, k):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.softmax = nn.Softmax(dim=-1)
        self.k = k

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        gate_logits = self.gate(x)  # (batch_size, seq_len, num_experts)
        gate_weights = self.softmax(gate_logits)

        # Select top-k experts
        top_k_weights, top_k_indices = torch.topk(gate_weights, self.k, dim=-1)
        # top_k_weights shape: (batch_size, seq_len, k)
        # top_k_indices shape: (batch_size, seq_len, k)

        # Normalize top-k weights to sum to 1
        top_k_weights_norm = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        return top_k_weights_norm, top_k_indices, gate_logits

# 4. Define MoE Layer
class MoELayer(nn.Module):
    def __init__(self, d_model, num_experts, k_experts):
        super().__init__()
        self.router = TopKRouter(d_model, num_experts, k_experts)
        # Experts with expansion factor of 4
        self.experts = nn.ModuleList([ExpertFFN(d_model, expansion_factor=4) for _ in range(num_experts)])
        self.d_model = d_model
        self.num_experts = num_experts
        self.k_experts = k_experts

    def forward(self, x, attention_mask=None):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # Get routing weights and indices
        top_k_weights_norm, top_k_indices, gate_logits = self.router(x)

        # Initialize output tensor
        output = torch.zeros_like(x, device=x.device)

        # Flatten x, top_k_weights_norm, top_k_indices for easier expert dispatch
        x_flat = x.view(-1, self.d_model)
        top_k_weights_norm_flat = top_k_weights_norm.view(-1, self.k_experts)
        top_k_indices_flat = top_k_indices.view(-1, self.k_experts)

        # Create a buffer for expert outputs
        expert_outputs_buffer = torch.zeros_like(x_flat)

        for i, expert_ffn in enumerate(self.experts):
            # Create a mask for tokens routed to expert i
            # A token is routed to expert i if i is in its top_k_indices
            expert_mask_for_current_expert = (top_k_indices_flat == i).any(dim=-1)

            if expert_mask_for_current_expert.any():
                tokens_for_expert = x_flat[expert_mask_for_current_expert]
                expert_output = expert_ffn(tokens_for_expert)

                # Get the weights corresponding to expert i for the tokens routed to it
                # This requires finding where 'i' appears in top_k_indices_flat and getting the corresponding weight
                indices_of_expert_i = (top_k_indices_flat[expert_mask_for_current_expert] == i).nonzero(as_tuple=True)
                assigned_weights = top_k_weights_norm_flat[expert_mask_for_current_expert][indices_of_expert_i[0], indices_of_expert_i[1]]

                # Accumulate weighted expert output
                expert_outputs_buffer[expert_mask_for_current_expert] += expert_output * assigned_weights.unsqueeze(-1)

        output = expert_outputs_buffer.view(batch_size, seq_len, self.d_model)

        return output, gate_logits, top_k_indices


# 5. Define MambaMoEClassifier Model
class MambaMoEClassifier(nn.Module):
    def __init__(self, d_model, num_layers, d_state, d_conv, num_experts, k_experts, num_labels_sentiment, num_labels_cot, vocab_size, max_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        # Pass d_conv to MambaBlock
        self.mamba_blocks = nn.ModuleList([MambaBlock(d_model, d_state, d_conv=d_conv) for _ in range(num_layers)])

        # Insert MoE layer after the second Mamba block (index 1 for 0-indexed list)
        self.moe_layer_index = 1
        self.moe_layer = MoELayer(d_model, num_experts, k_experts)

        self.sentiment_head = nn.Linear(d_model, num_labels_sentiment)
        self.cot_head = nn.Linear(d_model, num_labels_cot)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape

        # Embeddings
        x = self.embedding(input_ids)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = x + self.position_embedding(positions)

        # Mamba Blocks and MoE
        raw_gate_logits = None
        top_k_indices = None
        for i, block in enumerate(self.mamba_blocks):
            x = block(x) # x shape: (batch_size, seq_len, d_model)
            if i == self.moe_layer_index:
                x, raw_gate_logits, top_k_indices = self.moe_layer(x, attention_mask)

        # Global Average Pooling for classification heads
        # Only average over unmasked tokens
        if attention_mask is not None:
            # Expand attention mask to match d_model
            expanded_attention_mask = attention_mask.unsqueeze(-1).expand_as(x).float()
            masked_x = x * expanded_attention_mask
            # Sum along sequence dimension and divide by count of unmasked tokens
            pooled_output = masked_x.sum(dim=1) / expanded_attention_mask.sum(dim=1).clamp(min=1e-9) # Add clamp to avoid division by zero
        else:
            # If no attention mask, just average over sequence length
            pooled_output = x.mean(dim=1)

        # Classification heads
        sentiment_logits = self.sentiment_head(pooled_output)
        cot_logits = self.cot_head(pooled_output)

        return cot_logits, sentiment_logits, raw_gate_logits, top_k_indices

print("MambaBlock, ExpertFFN, TopKRouter, MoELayer, and MambaMoEClassifier classes defined with updated parameters.")

MambaBlock, ExpertFFN, TopKRouter, MoELayer, and MambaMoEClassifier classes defined with updated parameters.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MambaMoEClassifier(
    d_model=d_model,
    num_layers=num_layers,
    d_state=d_state,
    d_conv=d_conv,
    num_experts=num_experts,
    k_experts=k_experts,
    num_labels_sentiment=num_labels_sentiment,
    num_labels_cot=num_labels_cot,
    vocab_size=vocab_size,
    max_length=max_length
).to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# Loss functions
# For sentiment classification and CoT classification
criterion_sentiment = nn.CrossEntropyLoss()
criterion_cot = nn.CrossEntropyLoss()

# GradScaler for Automatic Mixed Precision (AMP)
scaler = GradScaler()

# Training Loop Parameters
num_epochs = 3 # As specified in the task

# Function to calculate MoE load balancing loss
def moe_load_balancing_loss(gate_logits, top_k_indices, num_experts, eps=1e-10):
    # gate_logits: (batch_size, seq_len, num_experts)
    # top_k_indices: (batch_size, seq_len, k)

    # Convert gate_logits to probabilities for all experts
    gate_probs = F.softmax(gate_logits, dim=-1)

    # Reshape gate_probs and top_k_indices to (total_tokens, num_experts) and (total_tokens, k)
    gate_probs_flat = gate_probs.view(-1, num_experts)
    top_k_indices_flat = top_k_indices.view(-1, k_experts)

    # Calculate P_i (mean gate probability for expert i across all tokens)
    P_i = gate_probs_flat.mean(dim=0)

    # Calculate C_i (fraction of tokens dispatched to expert i)
    C_i = torch.zeros(num_experts, device=gate_logits.device)
    for i in range(num_experts):
        C_i[i] = (top_k_indices_flat == i).any(dim=-1).float().mean()

    # Load balancing loss
    lb_loss = num_experts * torch.sum(P_i * C_i)

    return lb_loss


# Training and Evaluation Loop
def train_and_evaluate(model, train_dataloader, val_dataloader, optimizer, scaler,
                       criterion_sentiment, criterion_cot, alpha, beta, num_epochs, num_experts, k_experts, device):

    train_losses = []
    val_losses = []
    val_accuracies = []
    val_f1_scores = []
    val_expert_utilization_rates = []

    print("\nStarting training...")
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0

        for batch_idx, batch in enumerate(train_dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sentiment_labels = batch['label'].to(device)
            cot_pseudo_labels = batch['pseudo_labels'].to(device)

            optimizer.zero_grad()

            with autocast(): # Enable AMP
                cot_logits, sentiment_logits, raw_gate_logits, top_k_indices_batch = model(input_ids, attention_mask)

                # Calculate main sentiment loss
                main_loss = criterion_sentiment(sentiment_logits, sentiment_labels)

                # Calculate CoT auxiliary loss
                cot_loss = criterion_cot(cot_logits, cot_pseudo_labels)

                # Calculate MoE load balancing loss if gate_logits is available
                moe_lb_loss = 0.0
                if raw_gate_logits is not None and top_k_indices_batch is not None:
                    moe_lb_loss = moe_load_balancing_loss(raw_gate_logits.float(), top_k_indices_batch, num_experts, eps=1e-10)

                # Total loss
                loss = main_loss + (alpha * cot_loss) + (beta * moe_lb_loss)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_dataloader)}, Train Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)
        print(f"Epoch {epoch+1} training complete. Average Train Loss: {avg_train_loss:.4f}")

        # Validation Phase
        model.eval()
        total_val_loss = 0
        all_preds = []
        all_labels = []
        all_raw_gate_logits = [] # To collect raw gate logits
        all_top_k_indices = []   # To collect top_k_indices for MoE metrics

        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                sentiment_labels = batch['label'].to(device)
                cot_pseudo_labels = batch['pseudo_labels'].to(device)

                with autocast():
                    cot_logits, sentiment_logits, raw_gate_logits, top_k_indices_batch = model(input_ids, attention_mask)

                    main_loss = criterion_sentiment(sentiment_logits, sentiment_labels)
                    cot_loss = criterion_cot(cot_logits, cot_pseudo_labels)

                    moe_lb_loss = 0.0
                    if raw_gate_logits is not None and top_k_indices_batch is not None:
                         moe_lb_loss = moe_load_balancing_loss(raw_gate_logits.float(), top_k_indices_batch, num_experts, eps=1e-10)

                    loss = main_loss + (alpha * cot_loss) + (beta * moe_lb_loss)

                total_val_loss += loss.item()

                preds = torch.argmax(sentiment_logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(sentiment_labels.cpu().numpy())

                if raw_gate_logits is not None:
                    all_raw_gate_logits.append(raw_gate_logits.cpu())
                if top_k_indices_batch is not None:
                    all_top_k_indices.append(top_k_indices_batch.cpu())

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_losses.append(avg_val_loss)

        # Calculate metrics
        val_accuracy = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        val_accuracies.append(val_accuracy)
        val_f1_scores.append(val_f1)

        # Calculate MoE Expert Utilization Rate
        expert_utilization_rate = 0.0
        if len(all_top_k_indices) > 0: # Check if MoE was active and indices collected
            combined_top_k_indices = torch.cat(all_top_k_indices, dim=0)
            # Flatten to get all selected expert indices
            flat_top_k_indices = combined_top_k_indices.view(-1)
            # Get unique experts that were selected
            utilized_experts = torch.unique(flat_top_k_indices)
            expert_utilization_rate = (len(utilized_experts) / num_experts) * 100
        val_expert_utilization_rates.append(expert_utilization_rate)


        print(f"Epoch {epoch+1} validation complete. Average Val Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.4f}, F1-Macro: {val_f1:.4f}, Expert Utilization: {expert_utilization_rate:.2f}%")

    print("\nTraining complete!")
    return train_losses, val_losses, val_accuracies, val_f1_scores, val_expert_utilization_rates

# Run the training and evaluation loop
train_losses, val_losses, val_accuracies, val_f1_scores, val_expert_utilization_rates = train_and_evaluate(
    model, train_dataloader, val_dataloader, optimizer, scaler,
    criterion_sentiment, criterion_cot, alpha, beta, num_epochs, num_experts, k_experts, device
)

print("\nFinal Training and Validation Metrics:")
print(f"Train Losses: {train_losses}")
print(f"Validation Losses: {val_losses}")
print(f"Validation Accuracies: {val_accuracies}")
print(f"Validation F1-Macro Scores: {val_f1_scores}")
print(f"Validation Expert Utilization Rates: {val_expert_utilization_rates}")

Using device: cuda

Starting training...
Epoch 1/3, Batch 0/2105, Train Loss: 0.9463


/tmp/ipython-input-1521755744.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-1521755744.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Enable AMP


Epoch 1/3, Batch 100/2105, Train Loss: 0.9365
Epoch 1/3, Batch 200/2105, Train Loss: 0.9339
Epoch 1/3, Batch 300/2105, Train Loss: 0.9453
Epoch 1/3, Batch 400/2105, Train Loss: 0.9390
Epoch 1/3, Batch 500/2105, Train Loss: 0.9594
Epoch 1/3, Batch 600/2105, Train Loss: 0.9281
Epoch 1/3, Batch 700/2105, Train Loss: 0.8675
Epoch 1/3, Batch 800/2105, Train Loss: 0.8012
Epoch 1/3, Batch 900/2105, Train Loss: 0.7311
Epoch 1/3, Batch 1000/2105, Train Loss: 0.7808
Epoch 1/3, Batch 1100/2105, Train Loss: 0.7931
Epoch 1/3, Batch 1200/2105, Train Loss: 0.8000
Epoch 1/3, Batch 1300/2105, Train Loss: 0.7695
Epoch 1/3, Batch 1400/2105, Train Loss: 0.7571
Epoch 1/3, Batch 1500/2105, Train Loss: 0.7049
Epoch 1/3, Batch 1600/2105, Train Loss: 0.7179
Epoch 1/3, Batch 1700/2105, Train Loss: 0.7338
Epoch 1/3, Batch 1800/2105, Train Loss: 0.7887
Epoch 1/3, Batch 1900/2105, Train Loss: 0.7917
Epoch 1/3, Batch 2000/2105, Train Loss: 0.7027
Epoch 1/3, Batch 2100/2105, Train Loss: 0.7600
Epoch 1 training compl

/tmp/ipython-input-1521755744.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 validation complete. Average Val Loss: 0.7691, Accuracy: 0.5631, F1-Macro: 0.5406, Expert Utilization: 100.00%
Epoch 2/3, Batch 0/2105, Train Loss: 0.7143


...

Epoch 2 validation complete. Average Val Loss: 0.7283, Accuracy: 0.6376, F1-Macro: 0.6359, Expert Utilization: 100.00%
Epoch 3/3, Batch 0/2105, Train Loss: 0.6071


/tmp/ipython-input-1521755744.py:88: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Enable AMP


Epoch 3/3, Batch 100/2105, Train Loss: 0.6928
Epoch 3/3, Batch 200/2105, Train Loss: 0.6443
Epoch 3/3, Batch 300/2105, Train Loss: 0.6174
Epoch 3/3, Batch 400/2105, Train Loss: 0.6708
Epoch 3/3, Batch 500/2105, Train Loss: 0.6060
Epoch 3/3, Batch 600/2105, Train Loss: 0.6487
Epoch 3/3, Batch 700/2105, Train Loss: 0.7277
Epoch 3/3, Batch 800/2105, Train Loss: 0.6359
Epoch 3/3, Batch 900/2105, Train Loss: 0.6484
Epoch 3/3, Batch 1000/2105, Train Loss: 0.7387
Epoch 3/3, Batch 1100/2105, Train Loss: 0.6173
Epoch 3/3, Batch 1200/2105, Train Loss: 0.6599
Epoch 3/3, Batch 1300/2105, Train Loss: 0.5158
Epoch 3/3, Batch 1400/2105, Train Loss: 0.6489
Epoch 3/3, Batch 1500/2105, Train Loss: 0.7689
Epoch 3/3, Batch 1600/2105, Train Loss: 0.5637
Epoch 3/3, Batch 1700/2105, Train Loss: 0.7328
Epoch 3/3, Batch 1800/2105, Train Loss: 0.7424
Epoch 3/3, Batch 1900/2105, Train Loss: 0.7528
Epoch 3/3, Batch 2000/2105, Train Loss: 0.6644
Epoch 3/3, Batch 2100/2105, Train Loss: 0.7493
Epoch 3 training compl

/tmp/ipython-input-1521755744.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 3 validation complete. Average Val Loss: 0.6962, Accuracy: 0.6858, F1-Macro: 0.6828, Expert Utilization: 100.00%

Training complete!

Final Training and Validation Metrics:
Train Losses: [0.8192776217596548, 0.7168855550453386, 0.6515237575896845]
Validation Losses: [0.7691177576780319, 0.7282671928405762, 0.6961542878832135]
Validation Accuracies: [0.5630733944954128, 0.6376146788990825, 0.6857798165137615]
Validation F1-Macro Scores: [0.5405671140243523, 0.6358908222161146, 0.6828366926491515]
Validation Expert Utilization Rates: [100.0, 100.0, 100.0]


## Training Utilities, Low-Precision Training, and Training/Evaluation Loop

### Subtask:
Set up `AdamW` optimizer, loss functions (main, CoT, and MoE load-balancing with `α=0.2`, `β=0.05`), implement Automatic Mixed Precision (AMP/FP16), and define the training and evaluation loop. Evaluate the model on the test set, reporting Final Classification Accuracy, F1-Score (Macro), and an MoE Load Balancing Metric.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MambaMoEClassifier(
    d_model=d_model,
    num_layers=num_layers,
    d_state=d_state,
    d_conv=d_conv,
    num_experts=num_experts,
    k_experts=k_experts,
    num_labels_sentiment=num_labels_sentiment,
    num_labels_cot=num_labels_cot,
    vocab_size=vocab_size,
    max_length=max_length
).to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# Loss functions
# For sentiment classification and CoT classification
criterion_sentiment = nn.CrossEntropyLoss()
criterion_cot = nn.CrossEntropyLoss()

# GradScaler for Automatic Mixed Precision (AMP)
scaler = GradScaler()

# Training Loop Parameters
num_epochs = 3 # As specified in the task

# Function to calculate MoE load balancing loss
def moe_load_balancing_loss(gate_logits, top_k_indices, num_experts, eps=1e-10):
    # gate_logits: (batch_size, seq_len, num_experts)
    # top_k_indices: (batch_size, seq_len, k)

    # Convert gate_logits to probabilities for all experts
    gate_probs = F.softmax(gate_logits, dim=-1)

    # Calculate expert usage count (importance) and routing probabilities (load)
    expert_importance = torch.sum(gate_probs, dim=[0, 1]) # Sum over batch and sequence

    # Reshape gate_probs and top_k_indices to (total_tokens, num_experts) and (total_tokens, k)
    gate_probs_flat = gate_probs.view(-1, num_experts)
    top_k_indices_flat = top_k_indices.view(-1, k_experts)

    # Calculate P_i (mean gate probability for expert i across all tokens)
    P_i = gate_probs_flat.mean(dim=0)

    # Calculate C_i (fraction of tokens dispatched to expert i)
    C_i = torch.zeros(num_experts, device=gate_logits.device)
    for i in range(num_experts):
        C_i[i] = (top_k_indices_flat == i).any(dim=-1).float().mean()

    # Load balancing loss
    lb_loss = num_experts * torch.sum(P_i * C_i)

    return lb_loss


# Training and Evaluation Loop
def train_and_evaluate(model, train_dataloader, val_dataloader, optimizer, scaler,
                       criterion_sentiment, criterion_cot, alpha, beta, num_epochs, num_experts, k_experts, device):

    train_losses = []
    val_losses = []
    val_accuracies = []
    val_f1_scores = []
    val_expert_utilization_rates = []

    print("\nStarting training...")
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0

        for batch_idx, batch in enumerate(train_dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            sentiment_labels = batch['label'].to(device)
            cot_pseudo_labels = batch['pseudo_labels'].to(device)

            optimizer.zero_grad()

            with autocast(): # Enable AMP
                cot_logits, sentiment_logits, raw_gate_logits, top_k_indices_batch = model(input_ids, attention_mask)

                # Calculate main sentiment loss
                main_loss = criterion_sentiment(sentiment_logits, sentiment_labels)

                # Calculate CoT auxiliary loss
                cot_loss = criterion_cot(cot_logits, cot_pseudo_labels)

                # Calculate MoE load balancing loss if gate_logits is available
                moe_lb_loss = 0.0
                if raw_gate_logits is not None and top_k_indices_batch is not None:
                    moe_lb_loss = moe_load_balancing_loss(raw_gate_logits.float(), top_k_indices_batch, num_experts, eps=1e-10)

                # Total loss
                loss = main_loss + (alpha * cot_loss) + (beta * moe_lb_loss)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}/{len(train_dataloader)}, Train Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)
        print(f"Epoch {epoch+1} training complete. Average Train Loss: {avg_train_loss:.4f}")

        # Validation Phase
        model.eval()
        total_val_loss = 0
        all_preds = []
        all_labels = []
        all_raw_gate_logits = [] # To collect raw gate logits
        all_top_k_indices = []   # To collect top_k_indices for MoE metrics

        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                sentiment_labels = batch['label'].to(device)
                cot_pseudo_labels = batch['pseudo_labels'].to(device)

                with autocast():
                    cot_logits, sentiment_logits, raw_gate_logits, top_k_indices_batch = model(input_ids, attention_mask)

                    main_loss = criterion_sentiment(sentiment_logits, sentiment_labels)
                    cot_loss = criterion_cot(cot_logits, cot_pseudo_labels)

                    moe_lb_loss = 0.0
                    if raw_gate_logits is not None and top_k_indices_batch is not None:
                         moe_lb_loss = moe_load_balancing_loss(raw_gate_logits.float(), top_k_indices_batch, num_experts, eps=1e-10)

                    loss = main_loss + (alpha * cot_loss) + (beta * moe_lb_loss)

                total_val_loss += loss.item()

                preds = torch.argmax(sentiment_logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(sentiment_labels.cpu().numpy())

                if raw_gate_logits is not None:
                    all_raw_gate_logits.append(raw_gate_logits.cpu())
                if top_k_indices_batch is not None:
                    all_top_k_indices.append(top_k_indices_batch.cpu())

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_losses.append(avg_val_loss)

        # Calculate metrics
        val_accuracy = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        val_accuracies.append(val_accuracy)
        val_f1_scores.append(val_f1)

        # Calculate MoE Expert Utilization Rate
        expert_utilization_rate = 0.0
        if len(all_top_k_indices) > 0: # Check if MoE was active and indices collected
            combined_top_k_indices = torch.cat(all_top_k_indices, dim=0)
            # Flatten to get all selected expert indices
            flat_top_k_indices = combined_top_k_indices.view(-1)
            # Get unique experts that were selected
            utilized_experts = torch.unique(flat_top_k_indices)
            expert_utilization_rate = (len(utilized_experts) / num_experts) * 100
        val_expert_utilization_rates.append(expert_utilization_rate)


        print(f"Epoch {epoch+1} validation complete. Average Val Loss: {avg_val_loss:.4f}, Accuracy: {val_accuracy:.4f}, F1-Macro: {val_f1:.4f}, Expert Utilization: {expert_utilization_rate:.2f}%")

    print("\nTraining complete!")
    return train_losses, val_losses, val_accuracies, val_f1_scores, val_expert_utilization_rates

# Run the training and evaluation loop
train_losses, val_losses, val_accuracies, val_f1_scores, val_expert_utilization_rates = train_and_evaluate(
    model, train_dataloader, val_dataloader, optimizer, scaler,
    criterion_sentiment, criterion_cot, alpha, beta, num_epochs, num_experts, k_experts, device
)

print("\nFinal Training and Validation Metrics:")
print(f"Train Losses: {train_losses}")
print(f"Validation Losses: {val_losses}")
print(f"Validation Accuracies: {val_accuracies}")
print(f"Validation F1-Macro Scores: {val_f1_scores}")
print(f"Validation Expert Utilization Rates: {val_expert_utilization_rates}")

Using device: cuda

Starting training...


/tmp/ipython-input-2393001602.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-2393001602.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Enable AMP


Epoch 1/3, Batch 0/2105, Train Loss: 0.9282
Epoch 1/3, Batch 100/2105, Train Loss: 0.9459
Epoch 1/3, Batch 200/2105, Train Loss: 0.9137
Epoch 1/3, Batch 300/2105, Train Loss: 0.9628
Epoch 1/3, Batch 400/2105, Train Loss: 0.9048
Epoch 1/3, Batch 500/2105, Train Loss: 0.8847
Epoch 1/3, Batch 600/2105, Train Loss: 0.9091
Epoch 1/3, Batch 700/2105, Train Loss: 0.9045
Epoch 1/3, Batch 800/2105, Train Loss: 0.8987
Epoch 1/3, Batch 900/2105, Train Loss: 0.8570
Epoch 1/3, Batch 1000/2105, Train Loss: 0.8477
Epoch 1/3, Batch 1100/2105, Train Loss: 0.7348
Epoch 1/3, Batch 1200/2105, Train Loss: 0.7800
Epoch 1/3, Batch 1300/2105, Train Loss: 0.8954
Epoch 1/3, Batch 1400/2105, Train Loss: 0.8020
Epoch 1/3, Batch 1500/2105, Train Loss: 0.7782
Epoch 1/3, Batch 1600/2105, Train Loss: 0.7677
Epoch 1/3, Batch 1700/2105, Train Loss: 0.7733
Epoch 1/3, Batch 1800/2105, Train Loss: 0.8204
Epoch 1/3, Batch 1900/2105, Train Loss: 0.7703
Epoch 1/3, Batch 2000/2105, Train Loss: 0.7180
Epoch 1/3, Batch 2100/210

/tmp/ipython-input-2393001602.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 validation complete. Average Val Loss: 0.7670, Accuracy: 0.5803, F1-Macro: 0.5534, Expert Utilization: 100.00%
Epoch 2/3, Batch 0/2105, Train Loss: 0.7110


...

Epoch 2 validation complete. Average Val Loss: 0.7161, Accuracy: 0.6606, F1-Macro: 0.6604, Expert Utilization: 100.00%
Epoch 3/3, Batch 0/2105, Train Loss: 0.7249


/tmp/ipython-input-2393001602.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Enable AMP


Epoch 3/3, Batch 100/2105, Train Loss: 0.5530
Epoch 3/3, Batch 200/2105, Train Loss: 0.6565
Epoch 3/3, Batch 300/2105, Train Loss: 0.9023
Epoch 3/3, Batch 400/2105, Train Loss: 0.6049
Epoch 3/3, Batch 500/2105, Train Loss: 0.5817
Epoch 3/3, Batch 600/2105, Train Loss: 0.5854
Epoch 3/3, Batch 700/2105, Train Loss: 0.5048
Epoch 3/3, Batch 800/2105, Train Loss: 0.5527
Epoch 3/3, Batch 900/2105, Train Loss: 0.6491
Epoch 3/3, Batch 1000/2105, Train Loss: 0.6160
Epoch 3/3, Batch 1100/2105, Train Loss: 0.7390
Epoch 3/3, Batch 1200/2105, Train Loss: 0.6204
Epoch 3/3, Batch 1300/2105, Train Loss: 0.5080
Epoch 3/3, Batch 1400/2105, Train Loss: 0.6331
Epoch 3/3, Batch 1500/2105, Train Loss: 0.5283
Epoch 3/3, Batch 1600/2105, Train Loss: 0.7638
Epoch 3/3, Batch 1700/2105, Train Loss: 0.5538
Epoch 3/3, Batch 1800/2105, Train Loss: 0.6206
Epoch 3/3, Batch 1900/2105, Train Loss: 0.6278
Epoch 3/3, Batch 2000/2105, Train Loss: 0.4808
Epoch 3/3, Batch 2100/2105, Train Loss: 0.7124
Epoch 3 training compl

/tmp/ipython-input-2393001602.py:136: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 3 validation complete. Average Val Loss: 0.6932, Accuracy: 0.6950, F1-Macro: 0.6944, Expert Utilization: 100.00%

Training complete!

Final Training and Validation Metrics:
Train Losses: [0.829144795377011, 0.7093643643227439, 0.6313294046423498]
Validation Losses: [0.7670311459473201, 0.7161306376968112, 0.6932352802583149]
Validation Accuracies: [0.5802752293577982, 0.6605504587155964, 0.694954128440367]
Validation F1-Macro Scores: [0.5533762111874254, 0.660436137071651, 0.6944333199839807]
Validation Expert Utilization Rates: [100.0, 100.0, 100.0]


In [ ]:
import torch
from transformers import BertTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
import numpy as np

# Model Parameters
d_model = 128
num_layers = 4
d_state = 32
d_conv = 4 # Local convolution window of 4
num_experts = 8 # Updated to 8 experts
k_experts = 2   # Top-k=2 experts
num_labels_sentiment = 2  # SST-2 is binary classification
num_labels_cot = 2        # For short/long sequence pseudo-labeling (straightforward/extended reasoning)
vocab_size = None         # Will be set by tokenizer
max_length = 128
batch_size = 32

# Loss weights
alpha = 0.2  # CoT auxiliary loss weight
beta = 0.05  # MoE load-balancing loss weight (updated to 0.05)

# 1. Load SST-2 dataset
dataset = load_dataset('glue', 'sst2')

# 2. Load bert-base-uncased tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size

# First pass: Tokenize all training sentences to get lengths for median calculation
def get_sentence_lengths(examples):
    tokenized_inputs = tokenizer(examples['sentence'], truncation=False, padding=False)
    return {'lengths': [len(ids) for ids in tokenized_inputs['input_ids']]}

train_lengths_dataset = dataset['train'].map(get_sentence_lengths, batched=True, remove_columns=['sentence', 'label'])
median_length = int(np.median(train_lengths_dataset['lengths']))
print(f"Median sentence length for pseudo-labeling: {median_length}")

# Preprocessing function with median-based pseudo-labeling
def preprocess_function(examples, median_len):
    # Tokenize the text
    encoding = tokenizer(examples['sentence'], truncation=True, padding='max_length', max_length=max_length)

    # Derive pseudo-label for CoT based on median length
    # 'Straightforward' (0) if length <= median_len, 'Extended Reasoning' (1) if length > median_len
    # We use the original sentence length, not the tokenized length after truncation, for pseudo-labeling.
    # However, for consistency with 'truncation=True' for input_ids, let's use the tokenized length.
    # The instruction says 'sequences above median length', so it's based on the 'input sequence length'.
    # For simplicity, we'll use the tokenized length before padding to max_length for the pseudo-label.

    # Get original tokenized lengths (before padding/truncation to max_length for the model input)
    original_tokenized_lengths = [len(tokenizer.encode(s, add_special_tokens=True)) for s in examples['sentence']]

    pseudo_labels = [int(length > median_len) for length in original_tokenized_lengths]
    encoding['pseudo_labels'] = pseudo_labels
    return encoding

# Apply preprocessing using a lambda to pass median_length
encoded_dataset = dataset.map(lambda examples: preprocess_function(examples, median_length), batched=True)

# Convert to PyTorch tensors and format
encoded_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label', 'pseudo_labels'])

# 3. Create PyTorch DataLoaders
train_dataset = encoded_dataset['train']
val_dataset = encoded_dataset['validation']

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

print("Environment setup, data loading, preprocessing, and DataLoader creation complete with updated parameters.")
print(f"Vocab size: {vocab_size}")
print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of validation batches: {len(val_dataloader)}")

# Example of a batch
for batch in train_dataloader:
    print("\nExample batch structure:")
    print(f"  input_ids shape: {batch['input_ids'].shape}")
    print(f"  attention_mask shape: {batch['attention_mask'].shape}")
    print(f"  labels shape: {batch['label'].shape}")
    print(f"  pseudo_labels shape: {batch['pseudo_labels'].shape}")
    break

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Median sentence length for pseudo-labeling: 10


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Environment setup, data loading, preprocessing, and DataLoader creation complete with updated parameters.
Vocab size: 30522
Number of training batches: 2105
Number of validation batches: 28

Example batch structure:
  input_ids shape: torch.Size([32, 128])
  attention_mask shape: torch.Size([32, 128])
  labels shape: torch.Size([32])
  pseudo_labels shape: torch.Size([32])


In [ ]:
import torch
from transformers import BertTokenizer
import numpy as np


# Load the tokenizer (ensure it's the same one used for training)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Instantiate the model architecture (it needs to be the same as trained)
# We redefine it here in case the kernel was reset after training
demo_model = MambaMoEClassifier(
    d_model=d_model,
    num_layers=num_layers,
    d_state=d_state,
    d_conv=d_conv,
    num_experts=num_experts,
    k_experts=k_experts,
    num_labels_sentiment=num_labels_sentiment,
    num_labels_cot=num_labels_cot,
    vocab_size=vocab_size,
    max_length=max_length
).to(device)

# Load the saved state dictionary into the model
demo_model.load_state_dict(torch.load(model_save_path, map_location=device))
demo_model.eval() # Set model to evaluation mode

print(f"Model '{model_save_path}' loaded for demo.")

sentiment_labels = ["negative", "positive"]
cot_labels = ["straightforward", "extended reasoning"]

def predict_sentiment(text):
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        cot_logits, sentiment_logits, raw_gate_logits, top_k_indices = demo_model(input_ids, attention_mask)

    predicted_sentiment_idx = torch.argmax(sentiment_logits, dim=-1).item()
    predicted_cot_idx = torch.argmax(cot_logits, dim=-1).item()

    # Get MoE routing information for the first token (or average over sequence)
    # For simplicity, let's look at the routing for the first token in the sequence
    if raw_gate_logits is not None and top_k_indices is not None:
        # raw_gate_logits shape: (batch_size, seq_len, num_experts)
        # top_k_indices shape: (batch_size, seq_len, k)

        # Take the routing for the first token in the batch (index 0, first sequence token)
        first_token_gate_logits = raw_gate_logits[0, 0, :].cpu().numpy()
        first_token_top_k_indices = top_k_indices[0, 0, :].cpu().numpy()

        moe_info = {
            "gate_logits": first_token_gate_logits,
            "top_k_experts_indices": first_token_top_k_indices.tolist()
        }
    else:
        moe_info = "MoE routing info not available."

    return {
        "text": text,
        "predicted_sentiment": sentiment_labels[predicted_sentiment_idx],
        "predicted_cot_reasoning": cot_labels[predicted_cot_idx],
        "moe_routing_info": moe_info
    }

# --- Live Demo Examples ---
print("\n--- Live Demo ---")

example_texts = [
    "one of the greatest achievements in film history.",
    "Epic, absolute classic",
    "Takes the superhero format and sucks the life out of it",
    "Great acting, great plotline",
    "oh, what crap."
]

for i, text in enumerate(example_texts):
    result = predict_sentiment(text)
    print(f"\nExample {i+1}: '{result['text']}'")
    print(f"  Sentiment: {result['predicted_sentiment']}")
    print(f"  CoT Reasoning Type: {result['predicted_cot_reasoning']}")
    if isinstance(result['moe_routing_info'], dict):
        print(f"  Top {k_experts} Experts for first token: {result['moe_routing_info']['top_k_experts_indices']}")
    else:
        print(f"  MoE Routing Info: {result['moe_routing_info']}")

Model 'mamba_moe_sentiment_classifier.pth' loaded for demo.

--- Live Demo ---

Example 1: 'one of the greatest achievements in film history.'
  Sentiment: positive
  CoT Reasoning Type: extended reasoning
  Top 2 Experts for first token: [6, 1]

Example 2: 'Epic, absolute classic'
  Sentiment: positive
  CoT Reasoning Type: straightforward
  Top 2 Experts for first token: [6, 1]

Example 3: 'Takes the superhero format and sucks the life out of it'
  Sentiment: negative
  CoT Reasoning Type: extended reasoning
  Top 2 Experts for first token: [6, 1]

Example 4: 'Great acting, great plotline'
  Sentiment: positive
  CoT Reasoning Type: straightforward
  Top 2 Experts for first token: [6, 1]

Example 5: 'oh, what crap.'
  Sentiment: negative
  CoT Reasoning Type: straightforward
  Top 2 Experts for first token: [6, 1]
